# Pipeline ETL Completo

Extrae, Transforma, Carga de extremo a extremo

## Introducción

ETL (Extract, Transform, Load) y ELT (Extract, Load, Transform) son los dos patrones fundamentales para mover y procesar datos entre sistemas. ETL transforma los datos antes de cargarlos — ideal cuando el destino tiene recursos limitados o los datos deben estar limpios al llegar. ELT carga primero y transforma en destino — preferido cuando el destino es un data warehouse potente (BigQuery, Snowflake). Entender cuándo usar cada patrón y cómo construir pipelines robustos con manejo de errores y logging es la base del trabajo de un Data Engineer.

### Objetivos de Aprendizaje

- Distinguir ETL de ELT y saber cuándo usar cada patrón
- Diseñar y construir un pipeline ETL completo en Python
- Implementar manejo de errores con try/except en cada fase del pipeline
- Agregar logging profesional con el módulo logging de Python
- Comprender los conceptos de orquestación de pipelines (Airflow, etc.)

## ETL vs ELT: Cuándo Usar Cada Patrón

> ETL (Extract-Transform-Load) transforma los datos antes de cargarlos al destino. ELT (Extract-Load-Transform) carga los datos crudos primero y transforma en destino usando su potencia de cómputo. ETL es ideal para bases de datos relacionales clásicas, datos sensibles que deben limpiarse antes de persistirse, y cuando el volumen es manejable. ELT domina en data warehouses modernos en la nube (BigQuery, Snowflake, Redshift) que tienen capacidad masiva de procesamiento SQL.

In [ ]:
print("ETL: Transform BEFORE loading (control + cleanliness)")
print("ELT: Load THEN transform (scalability + warehouse power)")

print("""
ETL:  [Fuente] → EXTRAE → [Staging/Memoria] → TRANSFORMA → CARGA → [Destino]
      Los datos llegan limpios al destino.

ELT:  [Fuente] → EXTRAE → CARGA → [Data Warehouse] → TRANSFORMA (SQL/dbt)
      Los datos crudos llegan primero; la transformación ocurre en el destino.
""")

## Arquitectura del Pipeline ETL en Python

> Un pipeline ETL bien diseñado separa claramente las tres fases en funciones independientes. Cada función tiene una sola responsabilidad, puede ser testeada de forma aislada, y falla de forma controlada. La función principal orquesta las tres fases y gestiona errores globales. Esta separación de responsabilidades hace el pipeline más mantenible, testeable y reutilizable.

In [ ]:
import pandas as pd
import sqlite3
import logging
import io
import sys

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    stream=sys.stdout
)
logger = logging.getLogger(__name__)

CSV_DATOS = """id,fecha,producto,cantidad,precio,region
1,2024-01-15,Laptop,2,1200.00,Norte
2,2024-01-15,Mouse,10,25.50,Sur
3,2024-01-16,Laptop,1,1200.00,Norte
4,2024-01-16,Teclado,5,45.00,Este
2,2024-01-15,Mouse,10,25.50,Sur
5,2024-01-17,,3,0,Norte
6,2024-01-17,Monitor,2,350.00,Sur
7,2024-01-18,Laptop,1,1200.00,Oeste
"""

def extraer_datos(csv_texto: str) -> pd.DataFrame:
    logger.info("FASE 1 | Extrayendo datos...")
    df = pd.read_csv(io.StringIO(csv_texto))
    logger.info(f"  ✓ {len(df)} filas extraídas, {len(df.columns)} columnas")
    return df

def transformar_datos(df: pd.DataFrame) -> pd.DataFrame:
    logger.info("FASE 2 | Transformando datos...")
    df = df.copy()
    n_original = len(df)

    df.drop_duplicates(subset=['id'], keep='first', inplace=True)
    logger.info(f"  ✓ Duplicados eliminados: {n_original - len(df)}")

    df = df[df['producto'].notna() & (df['precio'] > 0)]
    logger.info(f"  ✓ Filas inválidas eliminadas: {n_original - len(df)} total")

    df['fecha'] = pd.to_datetime(df['fecha'])
    df['cantidad'] = df['cantidad'].astype(int)
    df['precio'] = df['precio'].astype(float)

    df['total_venta'] = (df['cantidad'] * df['precio']).round(2)
    df['anio_mes'] = df['fecha'].dt.to_period('M').astype(str)

    resumen = (
        df.groupby(['producto', 'region', 'anio_mes'])
        .agg(
            unidades_vendidas=('cantidad', 'sum'),
            ingresos_totales=('total_venta', 'sum'),
            num_transacciones=('id', 'count')
        )
        .reset_index()
        .round(2)
    )
    logger.info(f"  ✓ Datos transformados y agregados: {len(resumen)} filas de resumen")
    return resumen

def cargar_datos(df: pd.DataFrame, ruta_db: str = ":memory:") -> sqlite3.Connection:
    logger.info(f"FASE 3 | Cargando en SQLite ({ruta_db})...")
    conn = sqlite3.connect(ruta_db)
    df.to_sql('resumen_ventas', conn, if_exists='replace', index=False)
    n_cargadas = pd.read_sql("SELECT COUNT(*) as n FROM resumen_ventas", conn).iloc[0,0]
    logger.info(f"  ✓ {n_cargadas} filas verificadas en la tabla 'resumen_ventas'")
    return conn

def ejecutar_pipeline():
    logger.info("INICIANDO PIPELINE ETL — VENTAS")
    try:
        datos_crudos  = extraer_datos(CSV_DATOS)
        datos_limpios = transformar_datos(datos_crudos)
        conn          = cargar_datos(datos_limpios)
        logger.info("✅ PIPELINE COMPLETADO EXITOSAMENTE")

        resultado = pd.read_sql(
            "SELECT * FROM resumen_ventas ORDER BY ingresos_totales DESC",
            conn
        )
        print("\n=== RESULTADO FINAL ===")
        print(resultado.to_string(index=False))
        conn.close()
    except Exception as e:
        logger.error(f"❌ PIPELINE FALLIDO: {e}", exc_info=True)
        sys.exit(1)

ejecutar_pipeline()

## Manejo de Errores en Pipelines ETL

> Los pipelines en producción deben sobrevivir fallos parciales sin corromper datos. El patrón básico es: try/except en cada fase para capturar errores específicos, logging de errores con detalles completos (exc_info=True), y una estrategia de rollback o skip cuando una fila/archivo falla. Nunca dejes que un pipeline silenciosamente "se complete" con datos incorrectos.

In [ ]:
import pandas as pd
import sqlite3
import logging
import io

logger = logging.getLogger(__name__)

def extraer_con_validacion(ruta: str) -> pd.DataFrame:
    try:
        df = pd.read_csv(ruta, encoding='utf-8')
    except UnicodeDecodeError:
        logger.warning("UTF-8 falló, intentando con latin-1...")
        try:
            df = pd.read_csv(ruta, encoding='latin-1')
        except Exception as e:
            logger.error(f"No se pudo leer {ruta}: {e}")
            raise
    except FileNotFoundError:
        logger.error(f"Archivo no encontrado: {ruta}")
        raise
    except pd.errors.EmptyDataError:
        logger.error("El archivo CSV está vacío")
        raise

    columnas_requeridas = {'id', 'fecha', 'monto', 'categoria'}
    columnas_faltantes = columnas_requeridas - set(df.columns)
    if columnas_faltantes:
        raise ValueError(f"Columnas faltantes: {columnas_faltantes}")

    return df

def cargar_con_rollback(df: pd.DataFrame, ruta_db: str, tabla: str) -> None:
    conn = sqlite3.connect(ruta_db)
    try:
        conn.execute("BEGIN")
        df.to_sql(tabla, conn, if_exists='replace', index=False)
        conn.commit()
        logger.info(f"✓ Carga exitosa: {len(df)} filas en '{tabla}'")
    except Exception as e:
        conn.rollback()
        logger.error(f"Carga fallida, rollback ejecutado: {e}")
        raise
    finally:
        conn.close()

## Logging Profesional con el módulo logging

> El módulo logging de Python es la herramienta estándar para registrar la ejecución de pipelines. A diferencia de print(), logging permite niveles (DEBUG, INFO, WARNING, ERROR, CRITICAL), escribir a archivo y consola simultáneamente, incluir timestamps y contexto, y controlar la verbosidad sin modificar el código. En producción, los logs son la única forma de diagnosticar fallos post-mortem.

In [ ]:
import logging
import sys
from pathlib import Path

def configurar_logging(nivel: str = "INFO", archivo_log: str = None):
    nivel_num = getattr(logging, nivel.upper(), logging.INFO)
    formato = '%(asctime)s | %(name)s | %(levelname)-8s | %(message)s'
    fecha_fmt = '%Y-%m-%d %H:%M:%S'

    handlers = [logging.StreamHandler(sys.stdout)]

    if archivo_log:
        Path(archivo_log).parent.mkdir(parents=True, exist_ok=True)
        handlers.append(logging.FileHandler(archivo_log, encoding='utf-8'))

    logging.basicConfig(
        level=nivel_num,
        format=formato,
        datefmt=fecha_fmt,
        handlers=handlers
    )

configurar_logging("INFO")
logger = logging.getLogger("pipeline_etl")

logger.debug("Este mensaje solo aparece en modo DEBUG")
logger.info("Evento normal del flujo")
logger.warning("Algo inesperado pero no crítico")
logger.error("Algo falló pero el proceso puede continuar")
logger.critical("El proceso no puede continuar")

print("\nNiveles de logging:")
print("DEBUG    → detalles internos (solo desarrollo)")
print("INFO     → eventos normales del flujo")
print("WARNING  → algo inesperado pero no crítico")
print("ERROR    → algo falló pero el proceso puede continuar")
print("CRITICAL → el proceso no puede continuar en absoluto")

## Orquestación de Pipelines: Conceptos y Herramientas

> En producción, los pipelines deben ejecutarse automáticamente, respetar dependencias entre tareas, y reintentarse si fallan. Un orquestador es el sistema que gestiona esto. Apache Airflow es el estándar open source: define pipelines como DAGs (Directed Acyclic Graphs) en Python, con programación (schedule), reintentos automáticos, monitoreo visual y alertas. Para proyectos pequeños, cron + scripts Python es suficiente.

In [ ]:
print("""
DAG (Directed Acyclic Graph): El pipeline representado como grafo.
Cada nodo es una tarea. Las aristas son dependencias.

Ejemplo de DAG para un ETL de ventas:

[extraer_ventas] ──→ [transformar_ventas] ──→ [cargar_ventas]
                                              │
[extraer_clientes] ──→ [transformar_clientes] ──────→ [generar_reporte]

Las tareas de extracción pueden correr en PARALELO.
La carga espera que ambas transformaciones terminen.
El reporte espera que la carga esté completa.
""")

print("Orquestación: de cron simple a Airflow según la complejidad del proyecto")

## Pipeline ETL Completo: CSV → Limpieza → SQLite

Pipeline completo que lee un CSV de ventas, limpia y agrega los datos, y carga el resultado en SQLite con logging y manejo de errores.

In [ ]:
import pandas as pd
import sqlite3
import logging
import io
import sys

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    datefmt='%H:%M:%S',
    stream=sys.stdout
)
logger = logging.getLogger("etl_ventas")

CSV_DATOS = """id,fecha,producto,cantidad,precio,region
1,2024-01-15,Laptop,2,1200.00,Norte
2,2024-01-15,Mouse,10,25.50,Sur
3,2024-01-16,Laptop,1,1200.00,Norte
4,2024-01-16,Teclado,5,45.00,Este
2,2024-01-15,Mouse,10,25.50,Sur
5,2024-01-17,,3,0,Norte
6,2024-01-17,Monitor,2,350.00,Sur
7,2024-01-18,Laptop,1,1200.00,Oeste
"""

def extraer(csv_texto: str) -> pd.DataFrame:
    logger.info("FASE 1 | Extrayendo datos...")
    df = pd.read_csv(io.StringIO(csv_texto))
    logger.info(f"  ✓ Extraídas {len(df)} filas, {len(df.columns)} columnas")
    return df

def transformar(df: pd.DataFrame) -> pd.DataFrame:
    logger.info("FASE 2 | Transformando datos...")
    df = df.copy()
    n_original = len(df)

    df.drop_duplicates(subset=['id'], keep='first', inplace=True)
    logger.info(f"  ✓ Duplicados eliminados: {n_original - len(df)}")

    df = df[df['producto'].notna() & (df['precio'] > 0)]
    logger.info(f"  ✓ Filas inválidas eliminadas: {n_original - len(df)} total")

    df['fecha'] = pd.to_datetime(df['fecha'])
    df['cantidad'] = df['cantidad'].astype(int)
    df['precio'] = df['precio'].astype(float)

    df['total_venta'] = (df['cantidad'] * df['precio']).round(2)
    df['anio_mes'] = df['fecha'].dt.to_period('M').astype(str)

    resumen = (
        df.groupby(['producto', 'region', 'anio_mes'])
        .agg(
            unidades_vendidas=('cantidad', 'sum'),
            ingresos_totales=('total_venta', 'sum'),
            num_transacciones=('id', 'count')
        )
        .reset_index()
        .round(2)
    )
    logger.info(f"  ✓ Datos transformados y agregados: {len(resumen)} filas de resumen")
    return resumen

def cargar(df: pd.DataFrame, ruta_db: str = ":memory:") -> sqlite3.Connection:
    logger.info(f"FASE 3 | Cargando en SQLite ({ruta_db})...")
    conn = sqlite3.connect(ruta_db)
    df.to_sql('resumen_ventas', conn, if_exists='replace', index=False)
    n_cargadas = pd.read_sql("SELECT COUNT(*) as n FROM resumen_ventas", conn).iloc[0,0]
    logger.info(f"  ✓ {n_cargadas} filas verificadas en la tabla 'resumen_ventas'")
    return conn

def ejecutar_pipeline():
    logger.info("INICIANDO PIPELINE ETL — VENTAS")
    try:
        datos_crudos  = extraer(CSV_DATOS)
        datos_limpios = transformar(datos_crudos)
        conn          = cargar(datos_limpios)
        logger.info("✅ PIPELINE COMPLETADO EXITOSAMENTE")

        resultado = pd.read_sql(
            "SELECT * FROM resumen_ventas ORDER BY ingresos_totales DESC",
            conn
        )
        print("\n=== RESULTADO FINAL ===")
        print(resultado.to_string(index=False))
        conn.close()
    except Exception as e:
        logger.error(f"❌ PIPELINE FALLIDO: {e}", exc_info=True)
        sys.exit(1)

ejecutar_pipeline()

## Tips y Mejores Prácticas

> Diseña tus pipelines como funciones independientes (extraer, transformar, cargar) desde el principio. Esto te permite testear cada fase de forma aislada y reutilizar componentes en otros pipelines.

> Nunca modifiques el DataFrame original dentro de una función de transformación — siempre trabaja con df.copy(). Si el pipeline falla a mitad, quieres poder relanzar la transformación con los datos crudos intactos.

> Usa logging.INFO para eventos normales del flujo (conteos, tiempos, confirmaciones) y logging.ERROR con exc_info=True para errores — así el stack trace completo queda en el log y puedes diagnosticar fallos en producción.

> Para pipelines en producción, considera métricas de calidad: número de filas en entrada vs. salida, porcentaje de nulos, filas rechazadas por validación. Loggea estas métricas en cada ejecución para detectar anomalías.

## Errores Comunes

### Mezclar lógica de extracción, transformación y carga en una sola función

¿Por qué ocurre?
- Una función enorme que hace todo es imposible de testear, difícil de depurar cuando falla, y no reutilizable.

Solución
- Separa siempre las tres fases en funciones independientes. El orquestador llama a cada una en secuencia.

### No verificar el resultado de la carga

¿Por qué ocurre?
- to_sql() puede completarse sin errores pero haber cargado 0 filas si el DataFrame estaba vacío. Sin verificación, el pipeline reporta éxito con datos vacíos.

Solución
- Después de cargar, siempre ejecuta una consulta de verificación y compara con el número esperado de filas.

### Usar print() en vez de logging para monitorear pipelines

¿Por qué ocurre?
- print() no tiene niveles, no escribe a archivo automáticamente, no incluye timestamps, y no se puede desactivar sin modificar el código.

Solución
- Configura logging.basicConfig() al inicio de cada script y usa logger.info(), logger.warning(), logger.error() consistentemente.

### No manejar errores de encoding al leer archivos CSV

¿Por qué ocurre?
- pd.read_csv() usa UTF-8 por defecto. Archivos de sistemas Windows o exportados de Excel frecuentemente usan latin-1 o cp1252, causando UnicodeDecodeError.

Solución
- Intenta primero con UTF-8; si falla, reintenta con encoding="latin-1". O usa chardet para detectar el encoding automáticamente.